In [1]:
import os
import random
from datetime import datetime, timedelta, timezone

from elasticsearch import Elasticsearch
from elasticsearch.helpers import bulk
from faker import Faker

In [2]:
def get_client(hosts, basic_auth):
    return Elasticsearch(hosts=hosts, basic_auth=basic_auth, verify_certs=False)

In [3]:
elasticsearch = get_client(["http://localhost:9200"], ("elastic", "elastic"))
print(elasticsearch.cluster.health()["status"], elasticsearch.cluster.health()["number_of_nodes"])

green 1


### Faker로 뉴스 데이터 생성

In [4]:
fake = Faker("ko_KR")
random.seed(42)

In [5]:
from typing import Any


def make_news_doc(i: int) -> dict:
    now = datetime.now(timezone.utc)
    published_at = now - timedelta(hours=random.randint(0, 72), minutes=random.randint(0, 59))
    title = fake.sentence(nb_words=8).rstrip(".")
    body = "\n".join(fake.paragraphs(nb=3))
    category = random.choice(["경제", "사회", "정치", "IT", "국제", "스포츠", "문화"])
    source = random.choice(["연합뉴스", "조선일보", "중앙일보", "한겨레", "경향신문", "전자신문", "블로터"])
    author = fake.name()
    return {
        "news_id": f"N{i:06d}",
        "title": title,
        "body": body,
        "category": category,
        "source": source,
        "author": author,
        "tags": random.sample(["AI", "반도체", "환율", "증시", "부동산", "스타트업", "노동", "교육", "기후"], k=3),
        "published_at": published_at.isoformat(),
        "ingested_at": now.isoformat(),
        "url": fake.url(),
    }


def swap_alias(elastic: Elasticsearch, alias_: str, index_name_: str) -> dict[str, Any]:
    old_indices: list[str] = []
    if elastic.indices.exists_alias(name=alias_):
        old_indices = list(elastic.indices.get_alias(name=alias_).keys())

    actions_ = []
    for old in old_indices:
        if old != index_name_:
            actions_.append({"remove": {"index": old, "alias": alias_}})

    actions_.append({"add": {"index": index_name_, "alias": alias_}})
    elastic.indices.update_aliases(actions=actions_)
    removed = [x for x in old_indices if x != index_name_]
    return {"alias": alias_, "new_index": index_name_, "removed": removed, "actions": actions_}


def keep_last_n_indices_after_alias_swap(es: Elasticsearch, base_prefix: str, alias: str, keep: int = 3) -> dict:
    pattern = f"{base_prefix}-*"
    protected = set()
    if es.indices.exists_alias(name=alias):
        alias_map = es.indices.get_alias(name=alias)
        protected = set(alias_map.keys())

    settings = es.indices.get_settings(index=pattern, expand_wildcards="open")
    candidates = []
    for index_name, meta in settings.items():
        if index_name.startswith("."):
            continue
        if index_name in protected:
            pass
        creation_date = meta["settings"]["index"].get("creation_date")
        creation_date = int(creation_date) if creation_date is not None else 0
        candidates.append((index_name, creation_date))

    if not candidates:
        return {"kept": [], "deleted": [], "protected": list(protected)}

    candidates.sort(key=lambda x: x[1], reverse=True)
    keep_set = set([name for name, _ in candidates[:keep]])
    keep_set |= protected
    delete_list = [name for name, _ in candidates if name not in keep_set]

    deleted = []
    for idx in delete_list:
        if idx in protected:
            continue
        if es.indices.exists(index=idx):
            es.indices.delete(index=idx)
            deleted.append(idx)

    kept = [name for name, _ in candidates if name in keep_set]
    return {"kept": kept, "deleted": deleted, "protected": list(protected)}

## 인덱스 생성

In [6]:
base_index = "news"
alias = "news_current"
index_name = f"{base_index}-{datetime.now().strftime('%Y%m%d%H%M%S')}"

index_body = {
    "settings": {
        "number_of_shards": 1,
        "number_of_replicas": 0,
    },
    "mappings": {
        "properties": {
            "news_id": {"type": "keyword"},
            "title": {"type": "text"},
            "body": {"type": "text"},
            "category": {"type": "keyword"},
            "source": {"type": "keyword"},
            "author": {"type": "keyword"},
            "tags": {"type": "keyword"},
            "published_at": {"type": "date"},
            "ingested_at": {"type": "date"},
            "url": {"type": "keyword"},
        }
    },
}

In [7]:
if elasticsearch.indices.exists(index=index_name):
    elasticsearch.indices.delete(index=index_name)

elasticsearch.indices.create(index=index_name, **index_body)

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'news-20260109121545'})

In [8]:
doc_count = 2000
actions = ({"_index": index_name, "_id": doc["news_id"], "_source": doc, } for doc in (make_news_doc(i + 1) for i in range(doc_count)))
bulk(elasticsearch, actions, refresh="wait_for")

(2000, [])

### 조회 (간단 검색)

In [9]:
response_all = elasticsearch.search(index=index_name, size=5, query={"match_all": {}}, sort=[{"published_at": {"order": "desc"}}], )
print("total:", response_all["hits"]["total"]["value"])

for hit in response_all["hits"]["hits"]:
    src = hit["_source"]
    print(src["news_id"], src["published_at"], src["category"], src["title"])

total: 2000
N000681 2026-01-09T03:15:46.695028+00:00 스포츠 Ut in corrupti facilis vitae
N000664 2026-01-09T03:13:46.694053+00:00 국제 Fugit tempora perspiciatis aperiam odit dolorum veritatis sint velit reiciendis
N001921 2026-01-09T03:09:48.786483+00:00 사회 Adipisci ab ducimus doloribus provident architecto blanditiis
N001192 2026-01-09T03:06:47.740131+00:00 정치 Quis vitae ipsum consequuntur necessitatibus rerum asperiores aperiam
N001119 2026-01-09T03:04:47.734798+00:00 경제 Dolore cupiditate fugiat cupiditate nesciunt modi


In [10]:
query_word = "Sunt ipsa debitis"
response = elasticsearch.search(
    index=index_name,
    size=5,
    query={
        "bool": {
            "must": [{"multi_match": {"query": query_word, "fields": ["title^2", "body"]}}],
            "filter": [{"terms": {"category": ["IT", "경제", "문화"]}}],
        }
    },
    sort=[{"published_at": {"order": "desc"}}],
)

In [11]:
print(f"\n[search] index={index_name}, query='{query_word}' top5")

for hit in response["hits"]["hits"]:
    src = hit["_source"]
    print(f"- {src['published_at']} [{src['category']}/{src['source']}] {src['title']} (id={src['news_id']})")


[search] index=news-20260109121545, query='Sunt ipsa debitis' top5
- 2026-01-09T03:04:47.734798+00:00 [경제/전자신문] Dolore cupiditate fugiat cupiditate nesciunt modi (id=N001119)
- 2026-01-09T02:20:46.685198+00:00 [문화/블로터] Quos numquam laboriosam laborum delectus (id=N000503)
- 2026-01-09T02:08:46.706308+00:00 [문화/전자신문] Eveniet esse enim mollitia nam et tempore (id=N000883)
- 2026-01-09T02:08:45.701084+00:00 [경제/전자신문] At asperiores ipsa facere (id=N000063)
- 2026-01-09T01:57:48.776147+00:00 [경제/조선일보] Dignissimos vel illum perspiciatis ipsa libero quaerat ex tempore (id=N001783)


### Alias 연결 (스왑 방식)

In [12]:
swap_result = swap_alias(elasticsearch, alias, index_name)
print(swap_result)

{'alias': 'news_current', 'new_index': 'news-20260109121545', 'removed': ['news-20260109120605'], 'actions': [{'remove': {'index': 'news-20260109120605', 'alias': 'news_current'}}, {'add': {'index': 'news-20260109121545', 'alias': 'news_current'}}]}


In [13]:
keep_result = keep_last_n_indices_after_alias_swap(elasticsearch, base_index, alias, keep=3)
print(keep_result)

{'kept': ['news-20260109121545', 'news-20260109120605', 'news-20260109115533'], 'deleted': ['news-20260109115047'], 'protected': ['news-20260109121545']}


### Alias로 조회 (운영에서는 항상 alias로 조회)

In [15]:
response_alias = elasticsearch.search(index=alias, size=5, query={"match": {"body": "perspiciatis"}}, sort=[{"published_at": {"order": "desc"}}])

for hit in response_alias["hits"]["hits"]:
    src = hit["_source"]
    print(f"- {src['published_at']} [{src['category']}] {src['title']} (id={src['news_id']})")

- 2026-01-09T03:13:46.694053+00:00 [국제] Fugit tempora perspiciatis aperiam odit dolorum veritatis sint velit reiciendis (id=N000664)
- 2026-01-09T01:59:45.718284+00:00 [스포츠] Nobis earum fugiat voluptatum architecto dicta nostrum omnis (id=N000376)
- 2026-01-09T01:45:48.750800+00:00 [스포츠] Dignissimos facere libero enim tempora (id=N001513)
- 2026-01-09T01:39:45.705160+00:00 [정치] Error recusandae voluptatem cum nulla (id=N000133)
- 2026-01-09T01:37:47.758602+00:00 [스포츠] Similique nulla nemo accusamus cumque nisi rem magnam suscipit debitis (id=N001473)
